### День 1 — Базовая модель банковских счетов (усложнённый вариант)

🎯 Цель дня
Создать расширенную и абстрактную модель банковского счёта, способную служить базой для более сложных типов счетов.

📋 Требования

1. Абстрактный класс AbstractAccount
Создать абстрактный класс, содержащий:
- 🔑 уникальный идентификатор счёта
- 👤 данные владельца
- 💰 защищённый баланс
- 📊 статус счёта: активный, замороженный, закрытый
- 🔧 абстрактные методы:
  deposit(amount)
  withdraw(amount)
  get_account_info()

In [1]:
from dataclasses import dataclass
from abc import ABC, abstractmethod

'''
@dataclass cтрока-инструкция для Python: перед тем как окончательно создать
класс ниже, пропусти его через функцию dataclass
'''

@dataclass 
class AbstractAccount(ABC):
    account_id: str
    owner: str
    _balance: float
    status: str

    @abstractmethod 
    def deposit(self, amount): #положить деньги на счёт
        ...
        
    @abstractmethod
    def withdraw(self, amount): #снять деньги со счёта
        ...
        
    @abstractmethod
    def get_account_info(self): #показать выписку по счёту
        ...    

2. Класс BankAccount
Реализовать конкретный тип счёта с расширенными возможностями:
- ✅ валидация входящих данных
- 🔒 логические статусы и запрет операций при неверных статусах
- 🆔 автоматическая генерация короткого UUID при отсутствии номера счёта
- 💱 атрибут currency: RUB, USD, EUR, KZT, CNY

3. Исключения
Создать собственные классы ошибок:
- ❄️ AccountFrozenError счёт временно заблокирован
- 🚫 AccountClosedError счёта больше не существует
- ⚠️ InvalidOperationError то, что вы просите сделать, некорректно в принципе
- 💸 InsufficientFundsError денег не хватает

4. Базовые операции
Добавить проверки:
- ✅ корректность суммы
- 🔄 проверка статуса счета
- 🛡️ защита от отрицательных значений

5. Строковое представление
Метод __str__ должен показывать:
- 🏦 тип счета
- 👤 клиента
- 🔢 последние 4 цифры номера
- 📊 статус
- 💰 баланс и валюту



In [2]:
'''__init__ возьми параметры, которые пришли в BankAccount.__init__, 
и передай их дальше — пусть AbstractAccount.__init__ сделает свою обычную работу 
(сохранит их в self.account_id, self.owner и т.д.
'''
#исключения
class AccountFrozenError(Exception):
    '''Cчёт временно заблокирован'''
    pass

class AccountClosedError(Exception):
    '''Cчёт больше не существует'''
    pass

class InvalidOperationError(Exception):
    '''То, что вы просите сделать, некорректно в принципе'''
    pass

class InsufficientFundsError(Exception):
    '''Недостаточно средств на счёте'''
    pass

import uuid

class BankAccount(AbstractAccount):
    def __init__(self, owner, _balance, status, account_id=None, currency="RUB"):
        #валидация входящих данных
        if not owner:
            raise InvalidOperationError("Владелец счёта не может быть пустым")
        if not isinstance(_balance, (int, float)):
            raise InvalidOperationError(f"Баланс должен быть числом, получено: {type(_balance).__name__}")
        if _balance < 0:
            raise InvalidOperationError("Баланс не может быть отрицательным")
        if status not in ["active", "frozen", "closed"]:
            raise InvalidOperationError(f"Недопустимый статус: {status}")
        #автоматическая генерация короткого UUID при отсутствии номера счёта
        if account_id is None:
            account_id = uuid.uuid4().hex[:8]
        else:
            account_id = str(account_id)
            if not account_id:
                raise InvalidOperationError("Номер счёта не может быть пустым")
        if currency not in ["RUB", "USD", "EUR", "KZT", "CNY"]:
            raise InvalidOperationError(f"Недопустимая валюта: {currency}")
        super().__init__(account_id, owner, _balance, status)
        self.currency = currency

    def deposit(self, amount): #положить деньги на счёт
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(
                f"Операция разрешена только для активного счёта, текущий статус: {self.status}"
            )
        self._balance += amount   

    def withdraw(self, amount): #снять деньги со счёта
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(
                f"Операция разрешена только для активного счёта, текущий статус: {self.status}"
            )
        if self._balance - amount < 0:
            raise InsufficientFundsError("Недостаточно средств на счёте")
        self._balance -= amount

    def get_account_info(self):
        return {
            "owner": self.owner,
            "balance": self._balance,
            "status": self.status,
        }

    def __str__(self):
        return f"Тип счёта {type(self).__name__}, клиент {self.owner}, счёт ...{self.account_id[-4:]}, статус {self.status}, баланс {self._balance} {self.currency}"

In [3]:
acc = BankAccount("Иван", 1000.0, "active", currency="USD")
print(acc)

Тип счёта BankAccount, клиент Иван, счёт ...84f9, статус active, баланс 1000.0 USD


6. Тестирование
Создать демонстрацию:
- ➕ создание активного и замороженного счёта
- 🚫 попытка операций над замороженным счётом
- ✅ валидное пополнение и снятие

In [4]:
#Создание активного счёта
active_acc = BankAccount("Иван", 1000.0, "active", currency="RUB")
print(active_acc)

Тип счёта BankAccount, клиент Иван, счёт ...7f86, статус active, баланс 1000.0 RUB


In [5]:
#Создание замороженного счёта
frozen_acc = BankAccount("Мария", 500.0, "frozen", currency="USD")
print(frozen_acc)

Тип счёта BankAccount, клиент Мария, счёт ...21ee, статус frozen, баланс 500.0 USD


In [6]:
#Попытка пополнить замороженный счёт
try:
    frozen_acc.deposit(100)
except AccountFrozenError as e:
    print(f"Ошибка (ожидаемо): {e}")

Ошибка (ожидаемо): Счёт заморожен, операция запрещена


In [7]:
#Попытка снять деньги с того же замороженного счёта
try:
    frozen_acc.withdraw(50)
except AccountFrozenError as e:
    print(f"Ошибка (ожидаемо): {e}")

Ошибка (ожидаемо): Счёт заморожен, операция запрещена


In [8]:
#Попытка пополнить счёт со статусом blocked
blocked_acc = BankAccount("Иван", 1000.0, "active", currency="RUB")
blocked_acc.status = "blocked"
try:
    blocked_acc.deposit(100)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

Ошибка (ожидаемо): Операция разрешена только для активного счёта, текущий статус: blocked


In [9]:
#Попытка снять деньги с того же счёта со статусом blocked
try:
    blocked_acc.withdraw(50)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

Ошибка (ожидаемо): Операция разрешена только для активного счёта, текущий статус: blocked


In [10]:
#Валидное пополнение счёта
active_acc.deposit(500)
print(active_acc)

Тип счёта BankAccount, клиент Иван, счёт ...7f86, статус active, баланс 1500.0 RUB


In [11]:
#Валидное снятие счёта
active_acc.withdraw(200)
print(active_acc)

Тип счёта BankAccount, клиент Иван, счёт ...7f86, статус active, баланс 1300.0 RUB


In [12]:
#Номер передан числом — приводится к строке, print не падает
acc_num = BankAccount("Иван", 1000.0, "active", account_id=12345)
print(acc_num)

Тип счёта BankAccount, клиент Иван, счёт ...2345, статус active, баланс 1000.0 RUB


### День 2 — Базовая модель банковских счетов (усложнённый вариант)
🎯 Цель дня
Реализовать несколько дочерних классов счетов с расширенными возможностями.

📋 Требования

1. Наследование
Создать классы:
- SavingsAccount
- PremiumAccount
- InvestmentAccount

2. Функционал SavingsAccount
- 🔒 min_balance — минимальный остаток
- 📈 месячная ставка доходности
- 💰 метод apply_monthly_interest()

3. Функционал PremiumAccount
- ⬆️ увеличенные лимиты
- 💳 возможность овердрафта
- 📊 фиксированная комиссия

4. Функционал InvestmentAccount
- 📊 инвестиционные портфели
- 💼 виртуальные активы (stocks, bonds, etf)
- 📈 метод project_yearly_growth()

5. Полиморфизм
Каждый тип должен переопределять:
- withdraw()
- get_account_info()
- __str__()


In [13]:
class SavingsAccount(BankAccount):
    def __init__(self, owner, _balance, status, min_balance, interest_rate, account_id=None, currency="RUB"):
        super().__init__(owner, _balance, status, account_id, currency)
        if not isinstance(min_balance, (int, float)):
            raise InvalidOperationError(f"min_balance должен быть числом, получено: {type(min_balance).__name__}")
        if min_balance < 0:
            raise InvalidOperationError("min_balance не может быть отрицательным")
        if not isinstance(interest_rate, (int, float)):
            raise InvalidOperationError(f"interest_rate должен быть числом, получено: {type(interest_rate).__name__}")
        if interest_rate < 0:
            raise InvalidOperationError("interest_rate не может быть отрицательным")
        if _balance < min_balance:
            raise InvalidOperationError(
                f"Начальный баланс {_balance} не может быть меньше минимального остатка {min_balance}"
            )
        self.min_balance = min_balance
        self.interest_rate = interest_rate

    def apply_monthly_interest(self):
        interest = self._balance * self.interest_rate
        self._balance += interest
        return interest

    def withdraw(self, amount):
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(f"Операция разрешена только для активного счёта, текущий статус: {self.status}")
        if self._balance - amount < self.min_balance:
            raise InsufficientFundsError(f"Нельзя снять: баланс станет ниже минимального остатка {self.min_balance}")
        super().withdraw(amount)

    def get_account_info(self):
        info = super().get_account_info()
        info["min_balance"] = self.min_balance
        info["interest_rate"] = self.interest_rate
        return info

    def __str__(self):
        base = super().__str__()
        return f"{base}, мин. остаток {self.min_balance}, ставка {self.interest_rate}"

In [14]:
class PremiumAccount(BankAccount):
    def __init__(self, owner, _balance, status, overdraft_limit, transaction_fee, account_id=None, currency="RUB"):
        super().__init__(owner, _balance, status, account_id, currency)
        if not isinstance(overdraft_limit, (int, float)):
            raise InvalidOperationError(f"overdraft_limit должен быть числом, получено: {type(overdraft_limit).__name__}")
        if overdraft_limit < 0:
            raise InvalidOperationError("overdraft_limit не может быть отрицательным")
        if not isinstance(transaction_fee, (int, float)):
            raise InvalidOperationError(f"transaction_fee должен быть числом, получено: {type(transaction_fee).__name__}")
        if transaction_fee < 0:
            raise InvalidOperationError("transaction_fee не может быть отрицательным")
        self.overdraft_limit = overdraft_limit
        self.transaction_fee = transaction_fee
    
    def withdraw(self, amount):
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self._balance - amount - self.transaction_fee < -self.overdraft_limit:
            raise InsufficientFundsError("Превышен лимит овердрафта")
        self._balance -= (amount + self.transaction_fee)
   
    def get_account_info(self):
        info = super().get_account_info()
        info["overdraft_limit"] = self.overdraft_limit
        info["transaction_fee"] = self.transaction_fee
        return info

    def __str__(self):
        base = super().__str__()
        return f"{base}, овердрафт {self.overdraft_limit}, комиссия {self.transaction_fee}"

In [15]:
class InvestmentAccount(BankAccount):
    ASSET_GROWTH_RATES = {"stocks": 0.08, "bonds": 0.03, "etf": 0.05}

    def __init__(self, owner, _balance, status, portfolio=None, account_id=None, currency="RUB"):
        super().__init__(owner, _balance, status, account_id, currency)
        portfolio = portfolio if portfolio is not None else {}
        if not isinstance(portfolio, dict):
            raise InvalidOperationError(f"portfolio должен быть словарём, получено: {type(portfolio).__name__}")
        for asset_type, amount in portfolio.items():
            if asset_type not in self.ASSET_GROWTH_RATES:
                raise InvalidOperationError(
                    f"Недопустимый тип актива: {asset_type}. Разрешены: {list(self.ASSET_GROWTH_RATES.keys())}"
                )
            if not isinstance(amount, (int, float)):
                raise InvalidOperationError(f"Сумма актива '{asset_type}' должна быть числом, получено: {type(amount).__name__}")
            if amount < 0:
                raise InvalidOperationError(f"Сумма актива '{asset_type}' не может быть отрицательной")
        self.portfolio = portfolio

    def project_yearly_growth(self):
        total_growth = 0
        for asset_type, amount in self.portfolio.items():
            rate = self.ASSET_GROWTH_RATES.get(asset_type, 0)
            total_growth += amount * rate
        return total_growth

    def withdraw(self, amount):
        if not isinstance(amount, (int, float)):
            raise InvalidOperationError(f"Сумма должна быть числом, получено: {type(amount).__name__}")
        if amount <= 0:
            raise InvalidOperationError("Сумма операции должна быть положительной")
        if self.status == "frozen":
            raise AccountFrozenError("Счёт заморожен, операция запрещена")
        if self.status == "closed":
            raise AccountClosedError("Счёт закрыт, операция запрещена")
        if self.status != "active":
            raise InvalidOperationError(f"Операция разрешена только для активного счёта, текущий статус: {self.status}")
        invested_total = sum(self.portfolio.values())
        free_balance = self._balance - invested_total
        if amount > free_balance:
            raise InsufficientFundsError(
                f"Нельзя снять: {invested_total} уже вложено в портфель, свободно только {free_balance}"
            )
        super().withdraw(amount)

    def get_account_info(self):
        info = super().get_account_info()
        info["portfolio"] = self.portfolio
        info["projected_yearly_growth"] = self.project_yearly_growth()
        return info

    def __str__(self):
        base = super().__str__()
        return f"{base}, портфель {self.portfolio}"



6. Тестирование
Создать несколько счетов каждого типа и выполнить различные операции.



In [16]:
print("=== 1. Создание счетов трёх типов ===")
savings = SavingsAccount("Иван", 1000.0, "active", min_balance=100.0, interest_rate=0.02)
premium = PremiumAccount("Мария", 1000.0, "active", overdraft_limit=500.0, transaction_fee=10.0)
invest = InvestmentAccount("Пётр", 10000.0, "active", portfolio={"stocks": 5000, "bonds": 3000, "etf": 2000})
print(savings)
print(premium)
print(invest)

=== 1. Создание счетов трёх типов ===
Тип счёта SavingsAccount, клиент Иван, счёт ...d1b3, статус active, баланс 1000.0 RUB, мин. остаток 100.0, ставка 0.02
Тип счёта PremiumAccount, клиент Мария, счёт ...bb54, статус active, баланс 1000.0 RUB, овердрафт 500.0, комиссия 10.0
Тип счёта InvestmentAccount, клиент Пётр, счёт ...2c9a, статус active, баланс 10000.0 RUB, портфель {'stocks': 5000, 'bonds': 3000, 'etf': 2000}


In [17]:
print("\n=== 2. SavingsAccount: начисление процентов и снятие с защитой min_balance ===")
savings.apply_monthly_interest()
print(f"После начисления процентов: {savings}")
try:
    savings.withdraw(950)
except InsufficientFundsError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 2. SavingsAccount: начисление процентов и снятие с защитой min_balance ===
После начисления процентов: Тип счёта SavingsAccount, клиент Иван, счёт ...d1b3, статус active, баланс 1020.0 RUB, мин. остаток 100.0, ставка 0.02
Ошибка (ожидаемо): Нельзя снять: баланс станет ниже минимального остатка 100.0


In [18]:
print("\n=== 3. PremiumAccount: снятие с уходом в овердрафт ===")
premium.withdraw(1300)
print(f"После снятия с овердрафтом: {premium}")


=== 3. PremiumAccount: снятие с уходом в овердрафт ===
После снятия с овердрафтом: Тип счёта PremiumAccount, клиент Мария, счёт ...bb54, статус active, баланс -310.0 RUB, овердрафт 500.0, комиссия 10.0


In [19]:
print("\n=== 4. InvestmentAccount: прогноз роста и защита вложенных средств ===")
print(f"Прогноз роста портфеля за год: {invest.project_yearly_growth()}")

try:
    invest.withdraw(9000)  # вложено 10000, свободно 0 — должна быть ошибка
except InsufficientFundsError as e:
    print(f"Ошибка (ожидаемо): {e}")




=== 4. InvestmentAccount: прогноз роста и защита вложенных средств ===
Прогноз роста портфеля за год: 590.0
Ошибка (ожидаемо): Нельзя снять: 10000 уже вложено в портфель, свободно только 0.0


In [20]:
print("\n=== 5. InvestmentAccount: валидное снятие свободного остатка ===")
invest2 = InvestmentAccount("Анна", 12000.0, "active", portfolio={"stocks": 5000, "bonds": 3000, "etf": 2000})
print(f"Свободный остаток: {invest2._balance - sum(invest2.portfolio.values())}")
invest2.withdraw(1000)  # вложено 10000, свободно 2000 → 1000 можно снять
print(f"После снятия: {invest2}")


=== 5. InvestmentAccount: валидное снятие свободного остатка ===
Свободный остаток: 2000.0
После снятия: Тип счёта InvestmentAccount, клиент Анна, счёт ...b44c, статус active, баланс 11000.0 RUB, портфель {'stocks': 5000, 'bonds': 3000, 'etf': 2000}


In [23]:
print("\n=== 6. Тест: начальный баланс ниже min_balance (замечание 2) ===")
try:
    bad_savings = SavingsAccount("Света", 50.0, "active", min_balance=100.0, interest_rate=0.02)
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 6. Тест: начальный баланс ниже min_balance (замечание 2) ===
Ошибка (ожидаемо): Начальный баланс 50.0 не может быть меньше минимального остатка 100.0


In [24]:
print("\n=== 7. Тест: недопустимый тип актива в портфеле (замечание 3) ===")
try:
    bad_portfolio = InvestmentAccount("Ксения", 5000.0, "active", portfolio={"crypto": 1000})
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

print("\n=== 8. Тест: нечисловая сумма актива в портфеле (замечание 3) ===")
try:
    bad_portfolio2 = InvestmentAccount("Ксения", 5000.0, "active", portfolio={"stocks": "много"})
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")

print("\n=== 9. Тест: отрицательная сумма актива в портфеле (замечание 3) ===")
try:
    bad_portfolio3 = InvestmentAccount("Ксения", 5000.0, "active", portfolio={"stocks": -500})
except InvalidOperationError as e:
    print(f"Ошибка (ожидаемо): {e}")


=== 7. Тест: недопустимый тип актива в портфеле (замечание 3) ===
Ошибка (ожидаемо): Недопустимый тип актива: crypto. Разрешены: ['stocks', 'bonds', 'etf']

=== 8. Тест: нечисловая сумма актива в портфеле (замечание 3) ===
Ошибка (ожидаемо): Сумма актива 'stocks' должна быть числом, получено: str

=== 9. Тест: отрицательная сумма актива в портфеле (замечание 3) ===
Ошибка (ожидаемо): Сумма актива 'stocks' не может быть отрицательной


In [25]:
print("\n=== 10. Тест: нечисловая сумма в withdraw дочерних классов (замечание 1) ===")
for acc, name in [(savings, "SavingsAccount"), (premium, "PremiumAccount"), (invest, "InvestmentAccount")]:
    try:
        acc.withdraw("сто")
    except InvalidOperationError as e:
        print(f"{name}: Ошибка (ожидаемо): {e}")


=== 10. Тест: нечисловая сумма в withdraw дочерних классов (замечание 1) ===
SavingsAccount: Ошибка (ожидаемо): Сумма должна быть числом, получено: str
PremiumAccount: Ошибка (ожидаемо): Сумма должна быть числом, получено: str
InvestmentAccount: Ошибка (ожидаемо): Сумма должна быть числом, получено: str
